In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# for dirname, _, filenames in os.walk("/kaggle/input"):
# for filename in filenames:
# print(os.path.join(dirname, filenames))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub

# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# !pip install jupyter-black
# %load_ext jupyter_black

In [3]:
TRAIN_LABELS_PATH = (
    "/kaggle/input/competitions/soil-grain-size-from-photos/Training_labels_updated.csv"
)
TRAIN_PPM_PATH = (
    "/kaggle/input/competitions/soil-grain-size-from-photos/ppm_updated.csv"
)

TRAIN_IMG_PATH = "/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated"
TEST_IMG_PATH = "/kaggle/input/competitions/soil-grain-size-from-photos/Test_All_Photos/Test_All_Photos"
SAMPLE_SUB_PATH = (
    "/kaggle/input/competitions/soil-grain-size-from-photos/sample_submission.csv"
)

In [4]:
df_labels = pd.read_csv(TRAIN_LABELS_PATH)
df_ppm = pd.read_csv(TRAIN_PPM_PATH)
df_labels.shape, df_ppm.shape

((24, 12), (5, 5))

In [5]:
df_labels.head()

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,F827,9.4904,19.3892,47.6257,89.5554,99.8965,99.9896,100.0000,100.0000,100.0000,100.0000,100.0
1,G190,5.2076,10.2425,23.7537,50.3113,75.8311,85.7233,90.9898,94.8969,98.5814,100.0000,100.0
2,H030,6.7833,10.9091,19.0599,37.3235,66.8349,96.4965,97.9530,99.5691,100.0000,100.0000,100.0
3,H037,0.9320,6.4500,16.3216,27.8019,34.6629,46.3652,62.7168,79.0523,89.7830,100.0000,100.0
4,H038,1.8218,5.1400,10.6113,18.2778,24.6420,35.0767,47.2345,63.3992,81.0851,95.7127,100.0


In [6]:
df_ppm.head()

,phone,camera,width,height,ppm
0,iPhone 14,iPhone 14,4032,3024,13.942
1,iPhone 16,iPhone 16,5712,4284,19.525
2,Motorola Edge,motorola edge 20,4000,1800,11.492
3,Motorola Edge 60 Fusion,Motorola Edge 60 fusion,4096,2304,12.465
4,Samsung A52,SM-A525F,9248,6936,26.330


In [7]:
camera_metadata = {}
mean_ppm = df_ppm['ppm'].mean()
std_ppm = df_ppm['ppm'].std()
print(f'mean ppm = {mean_ppm} and std ppm = {std_ppm}')

for _, row in df_ppm.iterrows():
    camera_metadata[row["phone"]] = {
        "aspect_ratio": row["width"] / row["height"],
        "ppm": (row["ppm"] - mean_ppm) / std_ppm
    }

camera_metadata

mean ppm = 16.7508 and std ppm = 6.192125539748043


{'iPhone 14': {'aspect_ratio': 1.3333333333333333,
  'ppm': np.float64(-0.4536083743732191)},
 'iPhone 16': {'aspect_ratio': 1.3333333333333333,
  'ppm': np.float64(0.44802063236477585)},
 'Motorola Edge': {'aspect_ratio': 2.2222222222222223,
  'ppm': np.float64(-0.8492721871097564)},
 'Motorola Edge 60 Fusion': {'aspect_ratio': 1.7777777777777777,
  'ppm': np.float64(-0.6921371300515317)},
 'Samsung A52': {'aspect_ratio': 1.3333333333333333,
  'ppm': np.float64(1.5469970591697295)}}

In [8]:
for s in os.listdir(TRAIN_IMG_PATH)[:5]:
    split = s.split("_")
    prefix, _id, suffix = "".join(split[:-2]), split[-2], split[-1]
    print(prefix, _id, suffix)

SamsungA52 H666 01.jpg
MotorolaEdge60fusion H374 02.jpg
MotorolaEdge60fusion H374 03.jpg
MotorolaEdge H368 04.jpg
SamsungA52 H038 01.jpg


In [9]:
mobile_img = {}

for s in os.listdir(TRAIN_IMG_PATH):
    split = s.split("_")
    prefix, _id, suffix = "_".join(split[:-2]), split[-2], split[-1]
    key = prefix + "_" + _id
    if key in mobile_img:
        mobile_img[key].append(s)
    else:
        mobile_img[key] = [s]

In [10]:
len(mobile_img), mobile_img["Motorola_Edge_60_fusion_H374"], mobile_img[
    "Motorola_Edge_H368"
]

(45,
 ['Motorola_Edge_60_fusion_H374_02.jpg',
  'Motorola_Edge_60_fusion_H374_03.jpg',
  'Motorola_Edge_60_fusion_H374_01.jpg'],
 ['Motorola_Edge_H368_04.jpg',
  'Motorola_Edge_H368_02.jpg',
  'Motorola_Edge_H368_03.jpg',
  'Motorola_Edge_H368_01.jpg'])

In [11]:
for s in os.listdir(TRAIN_IMG_PATH):
    split = s.split("_")
    prefix, _id, suffix = "".join(split[:-2]), split[-2], split[-1]
    if _id == "H666":
        print(prefix, _id, suffix)

SamsungA52 H666 01.jpg
MotorolaEdge H666 03.jpg
MotorolaEdge H666 02.jpg
MotorolaEdge H666 01.jpg


## Same soil img taken by different cameras above cell

In [12]:
from torch.utils.data import Dataset, DataLoader
import torch
from PIL import Image
from pathlib import Path
from collections import defaultdict
import re

In [13]:
class SoilDataset(Dataset):
    def __init__(self, image_dir, labels_df, transform=None):
        self.image_dir = Path(image_dir)
        self.transform = transform

        self.labels = labels_df.set_index("sample_id")

        # sample_id -> image paths
        self.sample_to_images = defaultdict(list)

        # mobile_img and camera_metadata are already defined globally
        for key, suffixes in mobile_img.items():
            arr = key.split("_")
            camera = f"{arr[0]} {arr[1]}"
            sample_id = arr[-1]
            # if camera not in camera_metadata:
            #     print(camera, sample_id)

            for suffix in suffixes:
                image_path = self.image_dir / suffix
                self.sample_to_images[sample_id].append((image_path, camera))

        # Only keep samples present in this dataframe
        self.sample_ids = [
            sample_id
            for sample_id in self.labels.index
            if sample_id in self.sample_to_images
        ]

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        image_paths = self.sample_to_images[sample_id]
        images = []
        cameras = []

        for image_path, camera in image_paths:
            image = Image.open(image_path).convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
            images.append(image)
            cameras.append(camera)

        # [N_images, C, H, W]
        images = torch.stack(images)
        # [N_images, 2]
        metadata = torch.tensor(
            [
                [
                    camera_metadata[camera]["ppm"],
                    camera_metadata[camera]["aspect_ratio"],
                ]
                for camera in cameras
            ],
            dtype=torch.float32,
        )
        # [11]
        target = torch.tensor(self.labels.loc[sample_id].values, dtype=torch.float32)

        return {
            "sample_id": sample_id,
            "cameras": cameras,
            "metadata": metadata,
            "images": images,
            "target": target,
        }

In [14]:
sample_sub_df = pd.read_csv(SAMPLE_SUB_PATH)

sample_ids = []

for filename in os.listdir(TEST_IMG_PATH):

    name = os.path.splitext(filename)[0]

    # Remove image number at the end: (1), (2), ...
    name = re.sub(r"\s*\(\d+\)$", "", name)

    # Remove camera prefix
    name = re.sub(r"^iPhone14_", "", name)
    name = re.sub(r"^iPhone16_", "", name)
    name = re.sub(r"^iPhone_16_", "", name)

    # Match submission naming
    name = name.replace("HPC_Münster_BS6_9,0-10m", "HPC_Muenster_BS6_9_0-10m")

    sample_ids.append(name)

sample_ids = sorted(set(sample_ids))

In [15]:
sample_ids_set = set(sample_ids)
submission_ids_set = set(sample_sub_df["sample_id"])

print("In sample_ids but NOT in submission:")
print(sample_ids_set - submission_ids_set)

print("\nIn submission but NOT in sample_ids:")
print(submission_ids_set - sample_ids_set)

print("\nAll sample_ids present:", sample_ids_set <= submission_ids_set)
print("Same IDs:", sample_ids_set == submission_ids_set)

In sample_ids but NOT in submission:
set()

In submission but NOT in sample_ids:
set()

All sample_ids present: True
Same IDs: True


In [16]:
def parse_test_filename(image_path):
    """
    Example:
        iPhone16_HPC_Airbus BS10-4bis7 (2).JPG
        ->
        camera    = iPhone16
        sample_id = HPC_Airbus BS10-4bis7
    """
    name = Path(image_path).stem

    # Remove (1), (2), ... from the end
    name = re.sub(r"\s*\(\d+\)$", "", name)

    # Remove camera prefix
    if name.startswith("iPhone14_"):
        camera = "iPhone 14"
        sample_id = name[len("iPhone14_") :]

    elif name.startswith("iPhone16_"):
        camera = "iPhone 16"
        sample_id = name[len("iPhone16_") :]

    elif name.startswith("iPhone_16_"):
        camera = "iPhone 16"
        sample_id = name[len("iPhone_16_") :]

    else:
        # Useful for other cameras if they appear
        camera, sample_id = name.split("_", 1)

    # Match sample_submission naming
    sample_id = sample_id.replace("HPC_Münster_BS6_9,0-10m", "HPC_Muenster_BS6_9_0-10m")

    return camera, sample_id

In [17]:
class SoilTestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.transform = transform

        # sample_id -> list of image paths
        self.sample_to_images = defaultdict(list)

        # Parse all images
        for image_path in sorted(self.image_dir.iterdir()):
            # did not use camera for now
            camera, sample_id = parse_test_filename(image_path)
            # print(camera)
            self.sample_to_images[sample_id].append((image_path, camera))

        # One dataset item = one soil sample
        self.sample_ids = sorted(self.sample_to_images.keys())

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]

        image_data = self.sample_to_images[sample_id]

        images = []
        cameras = []
        metadata = []
        image_paths = []

        for image_path, camera in image_data:
            image = Image.open(image_path).convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
            images.append(image)
            cameras.append(camera)
            image_paths.append(str(image_path))

            metadata.append(
                [
                    camera_metadata[camera]["ppm"],
                    camera_metadata[camera]["aspect_ratio"],
                ]
            )

        # N_images x C x H x W
        images = torch.stack(images)
        # N_images, 2
        metadata = torch.tensor(metadata, dtype=torch.float32)

        return {
            "sample_id": sample_id,
            "images": images,
            "cameras": cameras,
            "metadata": metadata,
            "image_paths": image_paths,
        }

In [18]:
from torchvision import transforms

from sklearn.model_selection import train_test_split

In [19]:
IMG_SIZE = 384

train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(
            brightness=0.15, contrast=0.15, saturation=0.10, hue=0.03
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


test_val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


# test_transform = val_transform

In [20]:
train_df, val_df = train_test_split(
    df_labels, test_size=0.2, random_state=42, shuffle=True
)
train_df.shape, val_df.shape

((19, 12), (5, 12))

In [21]:
temp = SoilDataset(TRAIN_IMG_PATH, train_df, train_transform).__getitem__(0)
print(temp["sample_id"], temp["metadata"], temp["cameras"])

temp = SoilTestDataset(image_dir=TEST_IMG_PATH, transform=test_val_transform).__getitem__(0)
print(temp["sample_id"], temp["cameras"], temp["metadata"])
del temp

H367 tensor([[-0.8493,  2.2222],
        [-0.8493,  2.2222],
        [-0.8493,  2.2222],
        [ 1.5470,  1.3333]]) ['Motorola Edge', 'Motorola Edge', 'Motorola Edge', 'Samsung A52']
HPC_Airbus BS10-4bis7 ['iPhone 16', 'iPhone 16', 'iPhone 16'] tensor([[0.4480, 1.3333],
        [0.4480, 1.3333],
        [0.4480, 1.3333]])


In [22]:
train_dataset = SoilDataset(TRAIN_IMG_PATH, train_df, train_transform)
val_dataset = SoilDataset(TRAIN_IMG_PATH, val_df, test_val_transform)
test_dataset = SoilTestDataset(image_dir=TEST_IMG_PATH, transform=test_val_transform)

In [23]:
BATCH_SIZE = 1  # cause one id may have multiple images

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

In [25]:
len(train_loader), len(val_loader), len(test_loader)

(19, 5, 10)

In [26]:
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
# from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

In [27]:
class SoilModel(nn.Module):
    def __init__(self):
        super().__init__()

        weights = EfficientNet_V2_S_Weights.DEFAULT
        self.backbone = efficientnet_v2_s(weights=weights)

        feature_dim = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Identity()

        # caemra metadata
        self.metadata_net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )
        
        # +2 for aspect_ratio and ppm
        combined_dim = feature_dim + 32

        # Learn importance of each image
        self.attention = nn.Sequential(
            nn.Linear(combined_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        # Final prediction head
        # self.head = nn.Sequential(
        #     nn.Linear(combined_dim, 256),
        #     nn.ReLU(),
        #     nn.Dropout(0.5),
        #     nn.Linear(256, 11)
        # )
    
    def forward(self, x, metadata):
        B, N, C, H, W = x.shape

        # let say b=1,n=4
        # 4, 3, IMG_SIZE, IMG_SIZE
        x = x.view(B * N, C, H, W)

        # 4, 1280(efficient_net)
        x = self.backbone(x)
        # seperate out images (1, 4, 1280)
        x = x.view(B, N, -1)

        # Aggregate images belonging to same soil sample
        metadata = self.metadata_net(metadata)
        # [B, 1280 + 32]
        x = torch.cat([x, metadata], dim=-1)
        # [B, N, 1]
        attention_scores = self.attention(x)
        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        # x : [B, N, 1280 + 32]
        # attention_weights: [B, N, 1]
        # res -> [B, N, 1280 + 32]
        x = x * attention_weights
        # B,1280 + 32
        x = x.sum(dim=1)

        # [B, 11]
        # return self.head(x)
        # [B, 1280 + 32]
        return x

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SoilModel().to(device)

model.head, device

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 169MB/s]


(Sequential(
   (0): Linear(in_features=1312, out_features=256, bias=True)
   (1): ReLU()
   (2): Dropout(p=0.5, inplace=False)
   (3): Linear(in_features=256, out_features=11, bias=True)
 ),
 device(type='cuda'))

In [29]:
def emd_loss(logits, target):
    grain_sizes = torch.tensor(
        [0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2.0, 6.3, 20.0, 63.0, 200.0],
        dtype=logits.dtype,
        device=logits.device,
    )

    log_x = torch.log10(grain_sizes)

    dx = log_x[1:] - log_x[:-1]

    # logits -> positive bin masses
    mass = torch.softmax(logits, dim=-1)

    # masses -> cumulative distribution
    pred_cdf = torch.cumsum(mass, dim=-1) * 100.0

    # Difference between CDFs
    diff = torch.abs(pred_cdf[:, 1:] - target[:, 1:])

    # Weighted EMD
    loss = (diff * dx.unsqueeze(0)).sum(dim=-1)

    return loss.mean()

In [30]:
for param in model.backbone.parameters():
    param.requires_grad = False

for param in model.backbone.features[-2:].parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.backbone.features[-2:].parameters(),
            "lr": 1e-5
        },
        {
            "params": model.metadata_net.parameters(),
            "lr": 1e-3
        },
        {
            "params": model.attention.parameters(),
            "lr": 1e-3
        },
        {
            "params": model.head.parameters(),
            "lr": 1e-3
        },
    ],
    weight_decay=1e-4
)

In [31]:
total = sum(p.numel() for p in model.backbone.parameters())

trainable = sum(
    p.numel()
    for p in model.backbone.parameters()
    if p.requires_grad
)

print(f"Total:     {total:,}")
print(f"Trainable: {trainable:,}")

Total:     20,177,488
Trainable: 14,892,072


In [32]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    running_loss = 0.0

    for batch in loader:
        images = batch["images"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)
        metadata = batch["metadata"].to(device)

        optimizer.zero_grad()

        # [B, 11]
        logits = model(images, metadata)
        loss = emd_loss(logits, target)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [33]:
@torch.no_grad()
def validate(model, loader, device):
    model.eval()

    running_loss = 0.0
    for batch in loader:
        images = batch["images"].to(device)
        target = batch["target"].to(device)
        metadata = batch["metadata"].to(device)

        logits = model(images, metadata)
        loss = emd_loss(logits, target)

        running_loss += loss.item()

    return running_loss / len(loader)

In [47]:
from sklearn.linear_model import Ridge

In [34]:
num_epochs = 20

best_val_loss = float("inf")

for epoch in range(num_epochs):

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss = validate(model, val_loader, device)

    print(
        f"Epoch [{epoch+1:02d}/{num_epochs}] "
        f"Train EMD: {train_loss:.4f} | "
        f"Val EMD: {val_loss:.4f}"
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_soil_model.pth")
        print("  -> saved best model")

Epoch [01/20] Train EMD: 92.3598 | Val EMD: 110.3735
  -> saved best model
Epoch [02/20] Train EMD: 92.4453 | Val EMD: 102.6698
  -> saved best model
Epoch [03/20] Train EMD: 81.0848 | Val EMD: 101.3464
  -> saved best model
Epoch [04/20] Train EMD: 80.3526 | Val EMD: 101.8047
Epoch [05/20] Train EMD: 83.6807 | Val EMD: 100.2511
  -> saved best model
Epoch [06/20] Train EMD: 80.5314 | Val EMD: 102.3669
Epoch [07/20] Train EMD: 79.7567 | Val EMD: 101.7392
Epoch [08/20] Train EMD: 80.7776 | Val EMD: 102.1978
Epoch [09/20] Train EMD: 82.8192 | Val EMD: 102.2677
Epoch [10/20] Train EMD: 79.4759 | Val EMD: 102.5041
Epoch [11/20] Train EMD: 87.6034 | Val EMD: 102.0561
Epoch [12/20] Train EMD: 76.1454 | Val EMD: 101.8623
Epoch [13/20] Train EMD: 81.3060 | Val EMD: 102.6772
Epoch [14/20] Train EMD: 81.1705 | Val EMD: 102.8716
Epoch [15/20] Train EMD: 77.2542 | Val EMD: 104.9305
Epoch [16/20] Train EMD: 68.4264 | Val EMD: 99.5932
  -> saved best model
Epoch [17/20] Train EMD: 72.2796 | Val EMD:

In [36]:
from sklearn.model_selection import KFold
import numpy as np
import torch

K = 5

kf = KFold(n_splits=K, shuffle=True, random_state=42)
num_epochs = 100
patience = 30

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(df_labels)):
    print(f"\n{'='*40}")
    print(f"FOLD {fold + 1}/{K}")
    print(f"{'='*40}")

    # DataFrames
    fold_train_df = df_labels.iloc[train_idx].reset_index(drop=True)
    fold_val_df = df_labels.iloc[val_idx].reset_index(drop=True)

    print("Train:", len(fold_train_df))
    print("Val:", len(fold_val_df))

    train_dataset = SoilDataset(TRAIN_IMG_PATH, fold_train_df, train_transform)
    val_dataset = SoilDataset(TRAIN_IMG_PATH, fold_val_df, test_val_transform)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True
    )

    model = SoilModel().to(device)

    for param in model.backbone.parameters():
        param.requires_grad = False

    for param in model.backbone.features[-2:].parameters():
        param.requires_grad = True
    
    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.backbone.features[-2:].parameters(),
                "lr": 1e-5
            },
            {
                "params": model.metadata_net.parameters(),
                "lr": 1e-3
            },
            {
                "params": model.attention.parameters(),
                "lr": 1e-3
            },
            {
                "params": model.head.parameters(),
                "lr": 1e-3
            },
        ],
        weight_decay=1e-4
    )

    # Training
    best_val_loss = float("inf")
    
    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = validate(model, val_loader, device)

        print(
            f"Fold {fold+1} | "
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f}"
        )
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), f"best_fold_{fold+1}.pth")
            print("  -> saved best model")
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(
                f"  -> Early stopping at epoch {epoch+1}"
            )
            break

    fold_scores.append(best_val_loss)
    print(f"Fold {fold+1} BEST EMD: " f"{best_val_loss:.4f}")


FOLD 1/5
Train: 19
Val: 5
Fold 1 | Epoch [1/100] Train: 92.5395 | Val: 116.6181
  -> saved best model
Fold 1 | Epoch [2/100] Train: 82.0866 | Val: 106.6473
  -> saved best model
Fold 1 | Epoch [3/100] Train: 81.4161 | Val: 107.8829
Fold 1 | Epoch [4/100] Train: 84.8719 | Val: 111.0751
Fold 1 | Epoch [5/100] Train: 86.8687 | Val: 114.5045
Fold 1 | Epoch [6/100] Train: 78.3333 | Val: 112.3193
Fold 1 | Epoch [7/100] Train: 75.2466 | Val: 110.9502
Fold 1 | Epoch [8/100] Train: 70.7397 | Val: 107.7640
Fold 1 | Epoch [9/100] Train: 65.5568 | Val: 108.3543
Fold 1 | Epoch [10/100] Train: 63.1866 | Val: 107.4439
Fold 1 | Epoch [11/100] Train: 64.8290 | Val: 96.8818
  -> saved best model
Fold 1 | Epoch [12/100] Train: 53.7188 | Val: 84.3869
  -> saved best model
Fold 1 | Epoch [13/100] Train: 60.0419 | Val: 65.9933
  -> saved best model
Fold 1 | Epoch [14/100] Train: 49.3757 | Val: 59.4269
  -> saved best model
Fold 1 | Epoch [15/100] Train: 40.2562 | Val: 72.3120
Fold 1 | Epoch [16/100] Train:

In [37]:
print("\n========== RESULTS ==========")

for i, score in enumerate(fold_scores):
    print(f"Fold {i+1}: {score:.4f}")

print(f"\nMean CV EMD: {np.mean(fold_scores):.4f}")
print(f"Std CV EMD:  {np.std(fold_scores):.4f}")


========== RESULTS ==========
Fold 1: 25.5813
Fold 2: 49.0141
Fold 3: 67.5230
Fold 4: 40.1741
Fold 5: 32.3752

Mean CV EMD: 42.9335
Std CV EMD:  14.5726


In [38]:
best_fold = fold_scores.index(min(fold_scores)) + 1
f"best_fold_{best_fold}.pth"

'best_fold_1.pth'

In [39]:
best_model_path = f"best_fold_{best_fold}.pth"
model.load_state_dict(torch.load(best_model_path, map_location=device))

model = model.to(device)

In [40]:
val_loss = validate(model, val_loader, device)

print(f"best model Val emd loss: {val_loss:.4f}")

best model Val emd loss: 41.8684


In [41]:
# val_dataset = SoilDataset(TRAIN_IMG_PATH, df_labels[20:], test_val_transform)

# val_loader = DataLoader(
#     val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
# )

# val_loss = validate(model, val_loader, device)

# print(f"best model Val emd loss: {val_loss:.4f}")

In [42]:
predictions = []
sample_ids = []

model.eval()

with torch.no_grad():
    for batch in test_loader:
        images = batch["images"].to(device)
        metadata = batch["metadata"].to(device)
        logits = model(images, metadata)

        # logits -> probability mass
        mass = torch.softmax(logits, dim=-1)

        # mass -> cumulative distribution
        pred = torch.cumsum(mass, dim=-1) * 100.0
        predictions.extend(pred.cpu().numpy())
        sample_ids.extend(batch["sample_id"])

In [43]:
grain_columns = [
    "0.002",
    "0.0063",
    "0.02",
    "0.063",
    "0.2",
    "0.63",
    "2",
    "6.3",
    "20",
    "63",
    "200",
]

submission = pd.DataFrame(predictions, columns=grain_columns)
submission.insert(0, "sample_id", sample_ids)

In [44]:
submission.head(30)

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS10-4bis7,1.153071,2.263537,5.897902,12.558755,16.492119,23.630116,35.941097,55.269791,83.891273,96.886459,100.000000
1,HPC_Airbus BS12-5bis8,0.428403,0.875555,2.774240,7.583308,9.722708,16.797125,30.413263,53.338520,82.756317,98.335991,100.000000
2,HPC_Airbus BS6-3,0.196917,0.406898,1.498247,5.077019,6.776134,12.372513,27.933853,52.637291,85.893188,98.969337,100.000015
3,HPC_Audorfring,0.224055,0.522473,1.865484,5.192183,7.135302,12.983903,29.655462,53.222847,84.964584,98.380699,100.000015
4,HPC_Kleinkummerfeld 18-3,0.436635,0.787310,2.588705,6.567339,9.042755,14.880693,29.293833,50.376850,84.568123,98.325478,99.999985
5,HPC_Kleinkummerfeld 2-2,0.092167,0.223792,0.934061,3.598850,5.109725,10.500716,27.305275,51.316666,86.409332,99.252831,100.000000
6,HPC_Kleinkummerfeld 2-3,0.114073,0.240290,1.029266,3.488774,5.024660,10.650358,26.321121,50.291718,86.134163,99.230034,100.000000
7,HPC_Kleinkummerfeld 9-4,0.343017,0.683571,2.047513,6.132883,8.316300,15.197253,29.630714,52.232582,83.052734,98.358284,100.000000
8,HPC_Muenster_BS6_9_0-10m,0.592725,0.991081,3.168360,6.769067,9.001747,13.809048,28.456482,50.907219,82.530991,98.581100,100.000000
9,HPC_Testfeld Lidl WHV,0.398133,0.756719,2.857838,6.897141,8.257607,12.155194,25.528416,51.405750,86.956161,99.057030,100.000000


In [45]:
# Clip numerical floating point errors
submission[grain_columns] = submission[grain_columns].clip(0, 100)

# Force final point to exactly 100
submission["200"] = 100.0

# Check monotonicity
values = submission[grain_columns].values

print("Monotonic:", np.all(np.diff(values, axis=1) >= 0))

print("Final = 100:", np.all(submission["200"].values == 100.0))

print("Number of samples:", len(submission))
submission.head()

Monotonic: True
Final = 100: True
Number of samples: 10


,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS10-4bis7,1.153071,2.263537,5.897902,12.558755,16.492119,23.630116,35.941097,55.269791,83.891273,96.886459,100.0
1,HPC_Airbus BS12-5bis8,0.428403,0.875555,2.774240,7.583308,9.722708,16.797125,30.413263,53.338520,82.756317,98.335991,100.0
2,HPC_Airbus BS6-3,0.196917,0.406898,1.498247,5.077019,6.776134,12.372513,27.933853,52.637291,85.893188,98.969337,100.0
3,HPC_Audorfring,0.224055,0.522473,1.865484,5.192183,7.135302,12.983903,29.655462,53.222847,84.964584,98.380699,100.0
4,HPC_Kleinkummerfeld 18-3,0.436635,0.787310,2.588705,6.567339,9.042755,14.880693,29.293833,50.376850,84.568123,98.325478,100.0


In [46]:
submission.to_csv("submission.csv", index=False)